# Harman River Track Visualisation

Visualises the Harman River hiking track on an OSM basemap with an interactive AEDT time slider.

In [1]:
import geopandas as gpd
import pandas as pd
from zoneinfo import ZoneInfo
from ipyleaflet import Map, Polyline, CircleMarker, FullScreenControl, ScaleControl
import ipywidgets as widgets
from IPython.display import display

In [2]:
GPX_PATH = "../data/tracks/harman_river_track.gpx"
AEDT = ZoneInfo("Australia/Hobart")

# Load track points
gdf = gpd.read_file(GPX_PATH, layer="track_points")
gdf = gdf.sort_values("time").reset_index(drop=True)

# Convert UTC timestamps to AEDT
gdf["time_aedt"] = gdf["time"].dt.tz_convert(AEDT)

print(f"Track points : {len(gdf):,}")
print(f"Start (AEDT) : {gdf['time_aedt'].iloc[0].strftime('%Y-%m-%d %H:%M:%S %Z')}")
print(f"End   (AEDT) : {gdf['time_aedt'].iloc[-1].strftime('%Y-%m-%d %H:%M:%S %Z')}")

Track points : 18,340
Start (AEDT) : 2026-01-21 08:01:50 AEDT
End   (AEDT) : 2026-01-21 13:40:16 AEDT


In [3]:
# Resample to 1-minute intervals for the slider
# (keeps the full polyline intact; only the position marker steps by minute)
minute_gdf = (
    gdf.set_index("time")
    .resample("1min")
    .first()
    .dropna(subset=["geometry"])
    .reset_index()
)
minute_gdf["time_aedt"] = minute_gdf["time"].dt.tz_convert(AEDT)

# Pre-build slider labels and coordinate list
slider_labels = [
    t.strftime("%H:%M  %Z")
    for t in minute_gdf["time_aedt"]
]
minute_coords = [
    (row.geometry.y, row.geometry.x)
    for row in minute_gdf.itertuples()
]

print(f"Slider steps : {len(minute_coords)}  (one per minute)")

Slider steps : 309  (one per minute)


In [4]:
# Full track coordinates for the polyline
track_coords = [(row.geometry.y, row.geometry.x) for row in gdf.itertuples()]

# Map centred on track midpoint
mid_lat = (gdf.geometry.y.min() + gdf.geometry.y.max()) / 2
mid_lon = (gdf.geometry.x.min() + gdf.geometry.x.max()) / 2

m = Map(center=(mid_lat, mid_lon), zoom=13, scroll_wheel_zoom=True)
m.add(FullScreenControl())
m.add(ScaleControl(position="bottomleft"))

# Full track polyline
polyline = Polyline(
    locations=track_coords,
    color="#2271B3",
    weight=3,
    opacity=0.8,
)
m.add(polyline)

# Start / end markers
m.add(CircleMarker(
    location=track_coords[0],
    radius=7, color="green", fill_color="green", fill_opacity=1.0,
))
m.add(CircleMarker(
    location=track_coords[-1],
    radius=7, color="red", fill_color="red", fill_opacity=1.0,
))

# Current-position marker (moves with the slider)
pos_marker = CircleMarker(
    location=minute_coords[0],
    radius=10,
    color="white",
    fill_color="#E69F00",
    fill_opacity=1.0,
    weight=2,
)
m.add(pos_marker)

# ── Widgets ───────────────────────────────────────────────────────────────────
slider = widgets.SelectionSlider(
    options=list(zip(slider_labels, range(len(slider_labels)))),
    value=0,
    description="AEDT:",
    continuous_update=True,
    layout=widgets.Layout(width="95%"),
    style={"description_width": "50px"},
)

info_html = widgets.HTML(layout=widgets.Layout(margin="4px 0 0 8px"))

def update_info(idx):
    row = minute_gdf.iloc[idx]
    ele = f"{row['ele']:.0f} m" if pd.notna(row['ele']) else "—"
    info_html.value = f"<b>Elevation:</b> {ele}"

update_info(0)

def on_slider_change(change):
    idx = change["new"]
    pos_marker.location = minute_coords[idx]
    update_info(idx)

slider.observe(on_slider_change, names="value")

display(widgets.VBox(
    [m, slider, info_html],
    layout=widgets.Layout(width="100%"),
))